# KQA Pro memory-transfer continuation experiment

This notebook continues two existing MemorySplit checkpoints on KQA Pro without KoPL, SPARQL, chain-of-thought, or gold query supervision.

- **Dense:** fact values are learned in weights.
- **Split:** fact values remain loss-masked and are installed in the native organizer.
- **Main test:** answer reasoning questions whose required facts never appeared in training QA.

KoPL is used only offline to recover supporting facts, enforce leakage checks, and label reasoning skeletons for evaluation.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, shutil, subprocess, sys

drive.mount('/content/drive')

# ---- Edit these paths ----
DRIVE_ROOT = Path('/content/drive/MyDrive/memory-split')
DENSE_SOURCE_RUN = DRIVE_ROOT / 'models' / 'dense_run'
SPLIT_SOURCE_RUN = DRIVE_ROOT / 'models' / 'split_run'

# Repository and local hot-storage paths.
REPO_URL = 'https://github.com/syz2026/Memory-Split.git'
# Immutable code revision containing this experiment. Do not replace with a branch.
CODE_COMMIT = '5c881d982d11c8de12c41b963ac29ee455ea3e61'
REPO_DIR = Path('/content/Memory-Split')
LOCAL_ROOT = Path('/content/memory_split_kqa')
KQA_RAW = LOCAL_ROOT / 'raw'
KQA_PREPARED = LOCAL_ROOT / 'prepared'
KQA_CORPUS = LOCAL_ROOT / 'corpus'
CONFIG_DIR = LOCAL_ROOT / 'configs'
SOURCE_STAGE = LOCAL_ROOT / 'source_models'
RUN_ROOT = DRIVE_ROOT / 'kqa' / 'runs'

# Pilot settings. Scale only after the reports and smoke run pass.
TRAIN_QA = 20_000
DEV_QA = 1_000
TEST_QA = 2_000
NOISE_FACTS = 50_000
MAX_SUPPORT_FACTS = 16
MAX_VALUES_PER_LOOKUP = 16
CONTINUATION_TOKENS = 50_000_000
FACT_SHARE = 0.70
MIN_FACT_EXPOSURES = 2
SEED = 42

# Fixed conservative pilot LR. If running a sweep, select it only on eval_dev.jsonl
# in a separate development run before unblinding the transfer evaluation cells.
CONTINUATION_LR = 1e-4
TOKENS_PER_STEP = None
MICRO_BATCH_SIZE = 1
WARMUP_STEPS = None

for path in (LOCAL_ROOT, CONFIG_DIR, SOURCE_STAGE, RUN_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print('Dense source:', DENSE_SOURCE_RUN)
print('Split source:', SPLIT_SOURCE_RUN)
print('Durable run root:', RUN_ROOT)

In [ ]:
def run(command, cwd=None):
    print('+', ' '.join(map(str, command)), flush=True)
    subprocess.run([str(part) for part in command], cwd=cwd, check=True)

assert len(CODE_COMMIT) == 40 and all(c in '0123456789abcdef' for c in CODE_COMMIT), CODE_COMMIT
if not REPO_DIR.exists():
    run(['git', 'clone', REPO_URL, REPO_DIR])
run(['git', 'fetch', 'origin', CODE_COMMIT], cwd=REPO_DIR)
run(['git', 'checkout', '--detach', CODE_COMMIT], cwd=REPO_DIR)
code_commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True
).strip()
assert code_commit == CODE_COMMIT
print('Pinned code commit:', code_commit)

# Keep Colab's CUDA-matched torch wheel; install the remaining project deps.
requirements = []
for line in (REPO_DIR / 'requirements.txt').read_text().splitlines():
    stripped = line.strip()
    if stripped and not stripped.startswith('#') and not stripped.startswith('torch'):
        requirements.append(stripped)
run([sys.executable, '-m', 'pip', 'install', '-q', *requirements])

os.environ['PYTHONPATH'] = str(REPO_DIR)
os.chdir(REPO_DIR)
print('Repository:', REPO_DIR)
print('Python:', sys.version)

import torch
print('Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'

In [ ]:
import hashlib

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        while chunk := handle.read(8 * 1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()

source_hashes = {}

def validate_and_stage(label, drive_run):
    assert (drive_run / 'config.yaml').exists(), f'{label} config missing: {drive_run / "config.yaml"}'
    weights = drive_run / 'model.pt'
    if not weights.exists():
        weights = drive_run / 'ckpt.pt'
    assert weights.exists(), f'{label} model/checkpoint missing in {drive_run}'
    local_run = SOURCE_STAGE / label
    local_run.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_run / 'config.yaml', local_run / 'config.yaml')
    local_weights = local_run / weights.name
    source_sha = sha256_file(weights)
    if not local_weights.exists() or sha256_file(local_weights) != source_sha:
        print(f'Staging {label} weights from Drive ({weights.stat().st_size / 1e9:.2f} GB)...')
        shutil.copy2(weights, local_weights)
    assert sha256_file(local_weights) == source_sha
    (local_run / f'{local_weights.name}.sha256').write_text(source_sha + '\n')
    source_hashes[label] = source_sha
    print(f'{label} source SHA-256:', source_sha)
    return local_run

DENSE_LOCAL_RUN = validate_and_stage('dense', DENSE_SOURCE_RUN)
SPLIT_LOCAL_RUN = validate_and_stage('split', SPLIT_SOURCE_RUN)

def checkpoint_step(run_dir):
    path = run_dir / ('model.pt' if (run_dir / 'model.pt').exists() else 'ckpt.pt')
    try:
        state = torch.load(path, map_location='cpu', weights_only=False, mmap=True)
    except (TypeError, RuntimeError):
        state = torch.load(path, map_location='cpu', weights_only=False)
    assert isinstance(state, dict) and 'step' in state, f'checkpoint lacks step metadata: {path}'
    step = int(state['step'])
    del state
    return step

source_steps = [checkpoint_step(DENSE_LOCAL_RUN), checkpoint_step(SPLIT_LOCAL_RUN)]
assert source_steps[0] == source_steps[1], f'source checkpoint steps differ: {source_steps}'
print('Matched source checkpoint step:', source_steps[0])

usage = shutil.disk_usage('/content')
print('Local free disk GB:', round(usage.free / 1e9, 1))
assert usage.free > 10_000_000_000, 'At least 10 GB local free disk is recommended.'

## Download and prepare KQA Pro

The public test file has hidden labels, so the primary transfer split is constructed from labeled train/validation records. KoPL is executed only in this offline cell.

In [ ]:
run([
    sys.executable, 'scripts/download_kqa.py',
    '--out-dir', KQA_RAW,
], cwd=REPO_DIR)

run([
    sys.executable, 'scripts/prepare_kqa.py',
    '--dataset-dir', KQA_RAW,
    '--out-dir', KQA_PREPARED,
    '--train-limit', TRAIN_QA,
    '--dev-limit', DEV_QA,
    '--test-limit', TEST_QA,
    '--min-train-per-skeleton', 8,
    '--max-support-facts', MAX_SUPPORT_FACTS,
    '--max-values-per-lookup', MAX_VALUES_PER_LOOKUP,
    '--noise-limit', NOISE_FACTS,
    '--seed', SEED,
], cwd=REPO_DIR)

In [ ]:
split_report = json.loads((KQA_PREPARED / 'report.json').read_text())
print(json.dumps(split_report, indent=2))
assert all(split_report['checks'].values()), split_report['checks']
assert split_report['questions']['train'] == TRAIN_QA
assert split_report['questions']['dev'] == DEV_QA
assert split_report['questions']['test'] == TEST_QA
assert split_report['facts']['transfer'] > 0
print('Transfer split passed canonical semantic-atom leakage checks.')

## Build paired continuation shards

Both arms receive equal fact/QA token budgets. The builder refuses to continue unless every selected fact is exposed at least `MIN_FACT_EXPOSURES` times and every transfer lookup fits the model's native query-token cap.

In [ ]:
run([
    sys.executable, 'scripts/build_kqa_corpus.py',
    '--prepared-dir', KQA_PREPARED,
    '--out-dir', KQA_CORPUS,
    '--total-tokens', CONTINUATION_TOKENS,
    '--fact-share', FACT_SHARE,
    '--min-fact-exposures', MIN_FACT_EXPOSURES,
    '--seed', SEED,
], cwd=REPO_DIR)

corpus_report = json.loads((KQA_CORPUS / 'continuation_report.json').read_text())
print(json.dumps(corpus_report, indent=2))
assert all(corpus_report['checks'].values()), corpus_report['checks']

In [ ]:
# Evaluate source models only on seen-fact development QA. Transfer-test
# evaluation is deliberately deferred until after both continuation runs finish.
identity = (
    f"{code_commit[:8]}_{corpus_report['data_fingerprint'][:8]}_"
    f"d{source_hashes['dense'][:8]}_s{source_hashes['split'][:8]}"
)
SOURCE_DEV_ROOT = RUN_ROOT / f'source_dev_{identity}'
SOURCE_BASELINE_ROOT = RUN_ROOT / f'source_baselines_{identity}'

def evaluation_complete(out_dir, checkpoint_sha, eval_set):
    summary_path = out_dir / 'summary.json'
    if not summary_path.exists():
        return False
    summary = json.loads(summary_path.read_text())
    return (
        summary.get('checkpoint_sha256') == checkpoint_sha
        and summary.get('data_fingerprint') == corpus_report['data_fingerprint']
        and summary.get('eval_set') == eval_set
    )

for arm, source_run in [('dense', DENSE_LOCAL_RUN), ('split', SPLIT_LOCAL_RUN)]:
    dev_out = SOURCE_DEV_ROOT / arm
    if not evaluation_complete(dev_out, source_hashes[arm], 'dev'):
        run([
            sys.executable, 'scripts/run_kqa_evals.py',
            '--run', source_run,
            '--arm', arm,
            '--data-dir', KQA_CORPUS,
            '--out-dir', dev_out,
            '--eval-set', 'dev',
            '--batch-size', 16,
        ], cwd=REPO_DIR)
run([
    sys.executable, 'scripts/compare_kqa_runs.py',
    '--dense-run', SOURCE_DEV_ROOT / 'dense',
    '--split-run', SOURCE_DEV_ROOT / 'split',
    '--out', SOURCE_DEV_ROOT / 'comparison.json',
], cwd=REPO_DIR)
print('Transfer source baselines will be written after training:', SOURCE_BASELINE_ROOT)

## Create continued-training runs

`--init-from` loads only model weights from each source run. Optimizer state, data cursor, learning-rate schedule, and step count restart for the KQA continuation corpus. If a new KQA run already has `ckpt.pt`, `--resume auto` resumes that run instead.

In [ ]:
command = [
    sys.executable, 'scripts/make_kqa_configs.py',
    '--dense-run', DENSE_LOCAL_RUN,
    '--split-run', SPLIT_LOCAL_RUN,
    '--corpus-dir', KQA_CORPUS,
    '--out-root', RUN_ROOT,
    '--config-out', CONFIG_DIR,
    '--continuation-tokens', CONTINUATION_TOKENS,
    '--precision', 'auto',
]
if CONTINUATION_LR is not None:
    command += ['--lr', CONTINUATION_LR]
if WARMUP_STEPS is not None:
    command += ['--warmup-steps', WARMUP_STEPS]
if TOKENS_PER_STEP is not None:
    command += ['--tokens-per-step', TOKENS_PER_STEP]
if MICRO_BATCH_SIZE is not None:
    command += ['--micro-batch-size', MICRO_BATCH_SIZE]
run(command, cwd=REPO_DIR)

manifest = json.loads((CONFIG_DIR / 'kqa_manifest.json').read_text())
print(json.dumps(manifest, indent=2))

In [ ]:
# Continue the dense checkpoint. This can be rerun after a disconnect.
run([
    sys.executable, 'scripts/run_train.py',
    '--config', CONFIG_DIR / 'kqa_dense.yaml',
    '--resume', 'auto',
], cwd=REPO_DIR)

In [ ]:
# Continue the split checkpoint with its native masked-value mechanism.
run([
    sys.executable, 'scripts/run_train.py',
    '--config', CONFIG_DIR / 'kqa_split.yaml',
    '--resume', 'auto',
], cwd=REPO_DIR)

## Evaluate fact access and reasoning transfer

Each arm is first tested on direct access to exactly the transfer facts required by the selected QA. The main report includes unconditional answer accuracy and accuracy conditional on every supporting fact being available.

In [ ]:
DENSE_KQA_RUN = Path(manifest['dense']['out_dir'])
SPLIT_KQA_RUN = Path(manifest['split']['out_dir'])

continued_hashes = {
    run_dir: (run_dir / 'model.pt.sha256').read_text().strip()
    for run_dir in (DENSE_KQA_RUN, SPLIT_KQA_RUN)
}

# Score the completed continuations on development QA before unblinding test.
for run_dir in (DENSE_KQA_RUN, SPLIT_KQA_RUN):
    dev_out = run_dir / 'kqa_dev'
    if not evaluation_complete(dev_out, continued_hashes[run_dir], 'dev'):
        run([
            sys.executable, 'scripts/run_kqa_evals.py',
            '--run', run_dir,
            '--eval-set', 'dev',
            '--batch-size', 16,
        ], cwd=REPO_DIR)
run([
    sys.executable, 'scripts/compare_kqa_runs.py',
    '--dense-run', DENSE_KQA_RUN / 'kqa_dev',
    '--split-run', SPLIT_KQA_RUN / 'kqa_dev',
    '--out', RUN_ROOT / f'dev_comparison_{identity}.json',
], cwd=REPO_DIR)

# Hyperparameters are now fixed and both arms are trained, so unblind the
# transfer set for the source controls and continued models.
for arm, source_run in [('dense', DENSE_LOCAL_RUN), ('split', SPLIT_LOCAL_RUN)]:
    baseline_out = SOURCE_BASELINE_ROOT / arm
    if not evaluation_complete(baseline_out, source_hashes[arm], 'transfer'):
        run([
            sys.executable, 'scripts/run_kqa_evals.py',
            '--run', source_run,
            '--arm', arm,
            '--data-dir', KQA_CORPUS,
            '--out-dir', baseline_out,
            '--batch-size', 16,
        ], cwd=REPO_DIR)

run([
    sys.executable, 'scripts/compare_kqa_runs.py',
    '--dense-run', SOURCE_BASELINE_ROOT / 'dense',
    '--split-run', SOURCE_BASELINE_ROOT / 'split',
    '--out', SOURCE_BASELINE_ROOT / 'comparison.json',
], cwd=REPO_DIR)

for run_dir in (DENSE_KQA_RUN, SPLIT_KQA_RUN):
    eval_out = run_dir / 'kqa_evals'
    if evaluation_complete(eval_out, continued_hashes[run_dir], 'transfer'):
        print('Reusing completed evaluation:', run_dir)
        continue
    run([
        sys.executable, 'scripts/run_kqa_evals.py',
        '--run', run_dir,
        '--batch-size', 16,
    ], cwd=REPO_DIR)

In [ ]:
comparison_path = RUN_ROOT / (
    f"comparison_{DENSE_KQA_RUN.name}__{SPLIT_KQA_RUN.name}.json"
)
run([
    sys.executable, 'scripts/compare_kqa_runs.py',
    '--dense-run', DENSE_KQA_RUN,
    '--split-run', SPLIT_KQA_RUN,
    '--out', comparison_path,
], cwd=REPO_DIR)

comparison = json.loads(comparison_path.read_text())
print(json.dumps(comparison, indent=2))

## Interpretation

The primary endpoint is unconditional paired split-minus-dense transfer-QA accuracy. Conditional accuracy is descriptive only because conditioning on model-specific recall can select different subsets.

- If dense direct recall is low, a dense QA failure is primarily a storage failure.
- If dense recalls the required facts but loses on the symmetric direct-access diagnostic, the result is consistent with a utilization difference, not proof of one.
- If split organizer recall or QA-time support-query coverage is low, the experiment has a query-interface failure and cannot adjudicate the reasoning hypothesis.
- Compare source baselines, store-off collapse, wrong-store sensitivity, and equal oracle-context results before attributing a gain to externalized facts.
- Report by KQA skill, program length, and support count; many examples are short one-lookup cases.
- Repeat positive results across seeds, fact-load levels, and model sizes before making an efficiency claim.

Because the supplied source arms already diverged and this pilot introduces facts and QA together, it estimates a legacy end-to-end system difference. It does **not** identify external memory as the sole causal factor or isolate what the new answer-only QA taught. Add facts-only continuation controls before making that stronger claim.